# VocDenoiser — Colab driver

Thin driver that calls into the `vocdenoiser` package. Three stages:
1. **SNR filtering** — score clips, pick a clean training subset (CPU, numpy-only).
2. **Train** the β-VAE denoiser (GPU).
3. **Architecture search** — autoresearch-style loop over model families (GPU).

Runtime → Change runtime type → **GPU (A100/L4)** for stages 2–3.

In [ ]:
# Install (SNR-only: `pip install -e .`; with ML stack: `.[ml]`)
!git clone https://github.com/kalleknast/VocDenoiser.git
%cd VocDenoiser
!pip install -q -e '.[ml]'

In [ ]:
# Point at the clean phee-call set (never hardcoded). On Colab, mount Drive:
from google.colab import drive
drive.mount('/content/drive')
import os
# Copy to local disk first — do not train off a network mount.
os.environ['VOCDENOISER_DATA_ROOT'] = '/content/clean_calls'
!mkdir -p /content/clean_calls && rsync -a '/content/drive/MyDrive/phee_calls/' /content/clean_calls/

## 1. SNR filtering — select a clean training subset

In [ ]:
# Score every clip, then inspect the distribution before choosing a cutoff.
!python -m vocdenoiser.cli snr scan $VOCDENOISER_DATA_ROOT --out artifacts/snr_scores.csv --workers 8
!python -m vocdenoiser.cli snr report artifacts/snr_scores.csv --out-dir reports
# Optional: confirm the metric is call-agnostic on THIS data.
!python -m vocdenoiser.cli snr validate artifacts/snr_scores.csv \
    --src-dir $VOCDENOISER_DATA_ROOT --noise-dir data/Noise --noise-dir data/Cigarra --out-dir reports
from IPython.display import Markdown
Markdown(open('reports/snr_report.md').read())

In [ ]:
# Pick a threshold from the report (GMM crossover / Otsu / a percentile), then select.
!python -m vocdenoiser.cli snr select artifacts/snr_scores.csv \
    --src-dir $VOCDENOISER_DATA_ROOT --out artifacts/clean_manifest.csv \
    --keep-percentile 40 --link-dir /content/clean_subset

## 2. Train the β-VAE denoiser

In [ ]:
# Train on the clean subset selected above (noisy→clean pairs synthesised on the fly).
!python -m vocdenoiser.denoise.train \
    --data-root /content/clean_subset --max-epochs 100 --batch-size 32 --beta 4.0

In [ ]:
# Evaluate: UMAP of the 16-dim latents + RandomForest identity-preservation proxy.
!python -m vocdenoiser.denoise.eval \
    --data-root /content/clean_subset --ckpt checkpoints/last.ckpt --label-from parent

## 3. Architecture search (autoresearch-style)

Greedy hill-climb over a top-K frontier under a fixed compute budget; each
candidate trained for `--max-steps` and scored by spectrogram SI-SDR. The JSONL
ledger is the resumable state.

In [ ]:
# Dry-run the loop with NO GPU (synthetic landscape) to see the mechanics:
!python -m vocdenoiser.cli search run --harness mock --iters 30 --ledger artifacts/mock.jsonl
!python -m vocdenoiser.cli search report --ledger artifacts/mock.jsonl

In [ ]:
# Real search (GPU): trains each candidate for max-steps on the clean subset.
!python -m vocdenoiser.cli search run --harness torch \
    --data-root /content/clean_subset --iters 60 --max-steps 400 \
    --seeds 0 1 --ledger artifacts/search_ledger.jsonl
!python -m vocdenoiser.cli search report --ledger artifacts/search_ledger.jsonl